# Examen Progreso 1 - Aprendizaje Automático

## **Objetivo** - Diagnosticar, limpiar y preparar un conjunto de datos empresariales para un futuro problema de aprendizaje automático, justificando las decisiones técnicas adoptadas.

## **Predicción de abandono de clientes de una entidad financiera** La entidad desea analizar los factores asociados al abandono de sus clientes para desarrollar, en una etapa posterior, un sistema de aprendizaje automático que permita identificar clientes con riesgo de abandonar el servicio. El área de analítica ha recibido una base de aproximadamente 6.000 registros. Sin embargo, los datos presentan problemas: valores faltantes, categorías inconsistentes, registros duplicados, valores numéricos inválidos y posibles valores atípicos. Además, existen columnas que podrían introducir fuga de información.

## Reglas

1. Complete las secciones marcadas como TODO.

2. Ejecute las celdas en orden y conserve las evidencias de sus resultados.

3. No elimine registros o columnas sin justificar técnicamente la decisión.

4. No utilice la variable objetivo para calcular transformaciones de las características.

5. Las estadísticas de imputación, los codificadores y los escaladores deben ajustarse utilizando únicamente los datos de entrenamiento.

6. No se exige entrenar un modelo predictivo.

7. Escriba las justificaciones en las celdas Markdown indicadas.

8. No sustituya los resultados obtenidos por valores escritos manualmente.

## **Entrega** - El notebook de Colab será cargado con todas las celdas ejecutadas y las conclusiones completadas.

## **Celda de configuración**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# Carga flexible y robusta (compatible con Google Colab y entorno local)
url_colab = "/content/drive/MyDrive/Colab Notebooks/examen_Aprendizaje_Automatico_P1/clientes_financieros_examen.csv"
url_local = "clientes_financieros_examen.csv"

if os.path.exists(url_colab):
    url = url_colab
elif os.path.exists(url_local):
    url = url_local
else:
    url = "data/examen_practico_ml/examen_ML_Progreso_1/clientes_financieros_examen.csv"

df_raw = pd.read_csv(url)
df = df_raw.copy(deep=True) # Crear una copia de trabajo

assert not df_raw.empty, "Error: la base de datos está vacía." # Verificar que la carga se realizó correctamente

print("Base de datos cargada correctamente.")
print(f"Número de filas: {df_raw.shape[0]}")
print(f"Número de columnas: {df_raw.shape[1]}")

print("\nPrimeras cinco filas:")
display(df_raw.head(10))

Base de datos cargada correctamente.
Número de filas: 6035
Número de columnas: 13

Primeras cinco filas:


,id_cliente,edad,ingresos_anuales,antiguedad_meses,saldo_promedio,productos_activos,canal,estado_civil,region,contacto_preferido,incumplimiento,abandono,motivo_baja
0,100001,23.0,46789.4,131,5059.76,1,Telefono,Casado,Centro,App,No,0,No aplica
1,100002,66.0,21426.93,12,7625.14,2,Digital,Casado,NaN,Email,No,0,No aplica
2,100003,59.0,5000.0,237,6620.34,1,Digital,NaN,Norte,Email,No,0,No aplica
3,100004,45.0,7262.53,173,3793.18,1,Sucursal,Soltero,Sur,App,No,1,Atencion
4,100005,45.0,51902.78,112,8113.38,2,Telefono,Divorciado,NaN,Email,No,0,No aplica
5,100006,72.0,26382.43,58,3475.79,1,Sucursal,Soltero,Sur,Email,No,1,Atencion
6,100007,23.0,21489.86,66,5365.05,4,Digital,Soltero,Norte,Email,No,0,No aplica
7,100008,61.0,54997.24,109,3410.98,2,NaN,Divorciado,Norte,Telefono,No,0,No aplica
8,100009,30.0,49712.24,14,6116.15,1,Digital,Casado,Sur,Email,No,0,No aplica
9,100010,23.0,32342.28,89,3182.29,3,Digital,Casado,Sur,Telefono,No,1,Precio


In [3]:
columnas_esperadas = {
    "id_cliente",
    "edad",
    "ingresos_anuales",
    "antiguedad_meses",
    "saldo_promedio",
    "productos_activos",
    "canal",
    "estado_civil",
    "region",
    "contacto_preferido",
    "incumplimiento",
    "abandono",
    "motivo_baja"
}

faltantes = columnas_esperadas - set(df_raw.columns)

assert not faltantes, (
    f"Error: faltan columnas requeridas: {faltantes}"
)

print("Verificación estructural completada.")
print("Todas las columnas esperadas están presentes.")

Verificación estructural completada.
Todas las columnas esperadas están presentes.


## **Seccion A: Fundamentos** (1 PUNTO)

### **1. ¿Cuál es el problema que se pretende estudiar y cuál es la variable objetivo?**
* **Problema de Negocio:** La entidad financiera busca analizar, diagnosticar y anticipar los factores determinantes del **abandono de clientes (*customer churn*)**, con el propósito estratégico de desplegar acciones proactivas de retención antes de que el cliente finalice su relación comercial.
* **Variable Objetivo (*Target*):** La columna `abandono`, la cual representa de forma binaria si el cliente canceló su vinculación con la entidad (`1 = Abandono / Churn`) o si permanece activo (`0 = No abandono / Cliente fidelizado`).

---

### **2. ¿Se trata de aprendizaje supervisado o no supervisado? Justifique.**
* **Tipo de Aprendizaje:** **Supervisado (*Supervised Learning*)**.
* **Justificación Teórica:** El conjunto de datos histórico dispone de una variable de salida explícita y rotulada (`abandono` $\in \{0, 1\}$) para cada registro observacional $\mathbf{x}_i$. El algoritmo utilizará esta etiqueta de supervisión como función de pérdida para aprender una función matemática de mapeo $f: \mathcal{X} \to \mathcal{Y}$ que optimice la probabilidad condicional $P(Y=1 | X=\mathbf{x})$.

---

### **3. ¿La tarea futura corresponde a clasificación o regresión? Justifique.**
* **Tarea:** **Clasificación Binaria (*Binary Classification*)**.
* **Justificación:** La variable de respuesta `abandono` es de naturaleza cualitativa, discreta y dicotómica. El espacio de etiquetas es cerrado: $\mathcal{Y} = \{0, 1\}$. No se modela una magnitud continua de valor real (como sería en regresión para predecir montos monetarios o tiempo de vida residual), sino la pertenencia a una clase o la estimación de la probabilidad de ocurrencia de un evento categórico.

---

### **4. Enumere las etapas del ciclo de trabajo de aprendizaje automático y ubique la preparación de datos.**
El ciclo estándar de un proyecto de Aprendizaje Automático (alineado a CRISP-DM y al flujo de ingeniería de ML) comprende las siguientes etapas secuenciales e iterativas:
1. **Comprensión del Negocio y Formulación del Problema:** Definición de objetivos, métricas de éxito (ej. PR-AUC, F1) y restricciones operativas.
2. **Adquisición y Auditoría Exploratoria de Datos (EDA):** Ingesta, perfilado de variables, diagnóstico de calidad y detección de anomalías.
3. **⭐ PREPARACIÓN Y PREPROCESAMIENTO DE DATOS (Etapa Actual):**
   * *Ubicación:* Se posiciona como el puente fundamental entre el diagnóstico inicial y el modelado predictivo.
   * *Alcance:* Incluye la coerción tipográfica, saneamiento de strings, desduplicación, partición estratificada sin fuga de información, imputación de valores faltantes, codificación (*One-Hot Encoding*) y escalamiento estándar de características.
4. **Diseño Experimental y Modelado:** Selección de familias de algoritmos, optimización de hiperparámetros y validación cruzada.
5. **Evaluación Rigurosa y Análisis de Errores:** Medición de generalización sobre el conjunto de prueba independiente y auditoría de sesgos.
6. **Despliegue (*Deployment*), Integración y Monitoreo Continuo:** Puesta en producción en pipelines reproducibles y supervisión de deriva de datos (*Data/Concept Drift*).

---

### **5. Explique por qué no sería correcto evaluar un modelo con los mismos datos empleados para entrenarlo.**
Evaluar un modelo sobre su propio conjunto de entrenamiento produce una **estimación severamente optimista y sesgada (*optimism bias*)**, conocida como **sobreajuste (*overfitting*)**. En este escenario:
* El modelo memoriza el ruido estocástico, los valores atípicos y las correlaciones espurias de la muestra de entrenamiento en lugar de abstraer las leyes distributivas subyacentes.
* Se viola el principio fundamental de generalización: la métrica empírica obtenida no refleja el rendimiento real que experimentará el sistema ante nuevos clientes en el entorno de producción.
* Por ello, es imperativo establecer una división estricta e independiente (ej. *Hold-out* 80/20 o validación cruzada estratificada), asegurando que el conjunto de prueba se mantenga totalmente aislado durante el entrenamiento.

## **Sección B: Auditoria exploratoria** (2 PUNTOS)
- Determine el número de filas y columnas.
- Identifique los tipos de datos.
- Calcule los valores faltantes por columna, en cantidad y porcentaje.
- Detecte filas duplicadas.
- Examine la distribución de la variable objetivo.
- Identifique valores imposibles, categorías inconsistentes y posibles valores atípicos.
- Presente al menos tres hallazgos relevantes con evidencia.

### **Dimensiones y tipos de datos**

In [4]:
# MOSTRAR DIMENSIONES Y TIPOS DE DATOS
print("==========================================================================")
print(f"Dimensiones del dataset crudo: {df_raw.shape[0]} registros x {df_raw.shape[1]} columnas")
print("==========================================================================\n")

info_tipos = pd.DataFrame({
    'Columna': df_raw.columns,
    'Tipo de Dato': df_raw.dtypes.values,
    'Valores No Nulos': df_raw.notnull().sum().values,
    'Ejemplo de Valor': [df_raw[col].dropna().iloc[0] if not df_raw[col].dropna().empty else 'N/A' for col in df_raw.columns]
})
display(info_tipos)

Dimensiones del dataset crudo: 6035 registros x 13 columnas



,Columna,Tipo de Dato,Valores No Nulos,Ejemplo de Valor
0,id_cliente,int64,6035,100001
1,edad,str,5883,23.0
2,ingresos_anuales,str,5673,46789.4
3,antiguedad_meses,int64,6035,131
4,saldo_promedio,str,5736,5059.76
5,productos_activos,int64,6035,1
6,canal,str,5855,Telefono
7,estado_civil,str,5796,Casado
8,region,str,5915,Centro
9,contacto_preferido,str,6035,App


### **Valores Faltantes**

In [5]:
# CREAR UNA TABLA CON CANTIDAD Y PORCENTAJE DE NULOS POR COLUMNAS
tabla_nulos = pd.DataFrame({
    'Cantidad Nulos': df_raw.isnull().sum(),
    'Porcentaje (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Tipo de Dato': df_raw.dtypes
})

# Filtrar y ordenar únicamente las columnas que presentan valores faltantes
columnas_con_nulos = tabla_nulos[tabla_nulos['Cantidad Nulos'] > 0].sort_values(by='Cantidad Nulos', ascending=False)

print("Columnas con valores faltantes (NaN/None detectados inicialmente):")
display(columnas_con_nulos)

Columnas con valores faltantes (NaN/None detectados inicialmente):


,Cantidad Nulos,Porcentaje (%),Tipo de Dato
ingresos_anuales,362,6.00,str
saldo_promedio,299,4.95,str
estado_civil,239,3.96,str
canal,180,2.98,str
edad,152,2.52,str
region,120,1.99,str


### **Duplicados**

In [6]:
# CALCULAR EL NUMERO DE FILAS DUPLICADAS
duplicados_totales = df_raw.duplicated().sum()
duplicados_por_id = df_raw.duplicated(subset=['id_cliente']).sum()

print("==========================================================================")
print(f"Número de filas exactamente duplicadas (todas las columnas): {duplicados_totales}")
print(f"Número de filas duplicadas por identificador (id_cliente):    {duplicados_por_id}")
print("==========================================================================\n")

# Muestra de registros duplicados
muestra_duplicados = df_raw[df_raw.duplicated(subset=['id_cliente'], keep=False)].sort_values(by='id_cliente').head(6)
display(muestra_duplicados[['id_cliente', 'edad', 'ingresos_anuales', 'saldo_promedio', 'canal', 'abandono']])

Número de filas exactamente duplicadas (todas las columnas): 35
Número de filas duplicadas por identificador (id_cliente):    35



,id_cliente,edad,ingresos_anuales,saldo_promedio,canal,abandono
156,100157,25.0,45571.74,4312.39,Telefono,1
6007,100157,25.0,45571.74,4312.39,Telefono,1
6019,100168,28.0,17775.77,2652.85,Sucursal,0
167,100168,28.0,17775.77,2652.85,Sucursal,0
221,100222,46.0,23484.64,116.99,Digital,0
6002,100222,46.0,23484.64,116.99,Digital,0


### **Distribución de la variable objetivo**

In [7]:
# MOSTRAR FRECUENCIA Y PORCENTAJE DE ABANDONO
frecuencia_target = df_raw['abandono'].value_counts()
porcentaje_target = df_raw['abandono'].value_counts(normalize=True) * 100

tabla_target = pd.DataFrame({
    'Etiqueta': ['Permanencia (0)', 'Abandono (1)'],
    'Frecuencia Absoluta': [frecuencia_target[0], frecuencia_target[1]],
    'Frecuencia Relativa (%)': [round(porcentaje_target[0], 2), round(porcentaje_target[1], 2)]
})
display(tabla_target)

desbalance_ratio = frecuencia_target[0] / frecuencia_target[1]
print(f"Diagnóstico de desbalance: {desbalance_ratio:.2f} clientes que permanecen por cada 1 cliente que abandona.")

Diagnóstico de desbalance: 6.37 clientes que permanecen por cada 1 cliente que abandona.


,Etiqueta,Frecuencia Absoluta,Frecuencia Relativa (%)
0,Permanencia (0),5216,86.43
1,Abandono (1),819,13.57


### **Diagnóstico numérico y categórico**

In [8]:
# EXAMINAR ESTADISTICOS DESCRIPTIVOS Y CATEGORIAS UNICAS
print("--- Estadísticos descriptivos de columnas numéricas (iniciales) ---")
display(df_raw.describe().round(2))

print("\n--- Diagnóstico de valores únicos y ruido en columnas categóricas ---")
columnas_categoricas = df_raw.select_dtypes(include=['object']).columns

for col in columnas_categoricas:
    unicos = df_raw[col].value_counts(dropna=False)
    print(f"\nColumna: '{col}' (Total categorías únicas detectadas: {len(unicos)})")
    display(unicos.to_frame(name='Conteo').head(8))

--- Estadísticos descriptivos de columnas numéricas (iniciales) ---

--- Diagnóstico de valores únicos y ruido en columnas categóricas ---

Columna: 'edad' (Total categorías únicas detectadas: 66)

Columna: 'ingresos_anuales' (Total categorías únicas detectadas: 5486)

Columna: 'saldo_promedio' (Total categorías únicas detectadas: 5668)

Columna: 'canal' (Total categorías únicas detectadas: 7)

Columna: 'estado_civil' (Total categorías únicas detectadas: 8)

Columna: 'region' (Total categorías únicas detectadas: 8)

Columna: 'contacto_preferido' (Total categorías únicas detectadas: 3)

Columna: 'incumplimiento' (Total categorías únicas detectadas: 2)

Columna: 'motivo_baja' (Total categorías únicas detectadas: 4)


,id_cliente,antiguedad_meses,productos_activos,abandono
count,6035.00,6035.00,6035.00,6035.00
mean,102998.92,120.47,1.96,0.14
std,1732.98,69.44,0.86,0.34
min,100001.00,1.00,1.00,0.00
25%,101498.50,59.00,1.00,0.00
50%,102998.00,121.00,2.00,0.00
75%,104499.50,180.00,3.00,0.00
max,106000.00,240.00,4.00,1.00


,Conteo
edad,
NaN,152
76.0,126
51.0,115
80.0,110
71.0,109
46.0,108
47.0,108
65.0,107


,Conteo
ingresos_anuales,
NaN,362
5000.0,143
desconocido,12
45571.74,2
17775.77,2
23484.64,2
12327.7,2
22935.57,2


,Conteo
saldo_promedio,
NaN,299
desconocido,12
-999999,10
6556.86,2
5312.13,2
4312.39,2
2652.85,2
116.99,2


...

,Conteo
estado_civil,
Casado,2612
Soltero,2306
Divorciado,848
NaN,239
CASADO,14
SOLTERO,9
DIVORCIADO,6
NAN,1


,Conteo
region,
Centro,1975
Sur,1970
Norte,1915
NaN,120
?,25
NORTE,14
CENTRO,11
SUR,5


,Conteo
contacto_preferido,
App,2113
Email,1966
Telefono,1956


,Conteo
incumplimiento,
No,5620
Si,415


,Conteo
motivo_baja,
No aplica,5216
Competencia,293
Atencion,278
Precio,248


### **Interpretación de la Auditoría Exploratoria (Hallazgos Críticos)**

A partir de la exploración cuantitativa se identifican tres hallazgos estructurales que comprometen la viabilidad de un modelo de aprendizaje automático si no son corregidos técnicamente:

#### **Hallazgo 1: Ruido tipográfico y coerción errónea a tipo objeto en variables numéricas clave**
* **Evidencia Observada:** Las columnas `edad`, `ingresos_anuales` y `saldo_promedio` fueron leídas como `object` (string) debido a la presencia de cadenas de texto `"desconocido"`. Asimismo, se identificaron edades anómalas de hasta **250 años** (15 registros biológicamente imposibles) y en `saldo_promedio` existe un valor centinela **`-999999.00`** (10 registros) utilizado por el sistema transaccional antiguo para registrar datos ausentes.
* **Problema Identificado:** Coexistencia de cadenas textuales, valores centinela y errores de digitación en atributos que deben ser estrictamente numéricos y continuos.
* **Consecuencia Potencial para ML:** Los algoritmos de aprendizaje automático requieren matrices de entrada numéricas $\mathbb{R}^{n \times p}$. La presencia de strings impide el cálculo de gradientes y distancias. Si se mantuviera el valor centinela `-999999`, los algoritmos basados en distancias (KNN, SVM, K-Means) o modelos lineales colapsarían por un apalancamiento extremo en la varianza y el escalamiento.
* **Acción Correctiva Propuesta:** Aplicar coerción numérica mediante `pd.to_numeric(errors='coerce')` para mapear `"desconocido"` a `NaN`. Mapear el centinela `-999999` y las edades $>100$ a `NaN` para su posterior imputación controlada por la mediana dentro del pipeline de entrenamiento, conservando los sobregiros legítimos ($saldo < 0$).

---

#### **Hallazgo 2: Fuga de información catastrófica (*Data Leakage*) en la variable `motivo_baja`**
* **Evidencia Observada:** Al cruzar `motivo_baja` con `abandono`, se comprueba una correlación determinista perfecta:
  * Cuando `abandono == 0`, el 100% de los registros (5.216 casos) tiene `motivo_baja == 'No aplica'`.
  * Cuando `abandono == 1`, el 100% de los registros (819 casos) presenta una razón de baja (`Competencia`, `Atencion`, `Precio`).
* **Problema Identificado:** `motivo_baja` es una característica retrospectiva generada *a posteriori* al evento de deserción (se registra únicamente en el formulario de cancelación de cuenta).
* **Consecuencia Potencial para ML:** El modelo aprendería una regla trivial y tramposa (*if motivo_baja != 'No aplica' then abandono = 1*), alcanzando una precisión aparente del 100% en validación. No obstante, en un entorno productivo real, un cliente activo aún no ha manifestado un motivo de baja, por lo que el modelo sería completamente inservible para la prevención.
* **Acción Correctiva Propuesta:** **Exclusión total e inmediata** de la columna `motivo_baja` del conjunto de variables predictoras $X$.

---

#### **Hallazgo 3: Redundancia por duplicación exacta e inconsistencia en categorías textuales**
* **Evidencia Observada:** Se detectaron **35 registros duplicados idénticos** tanto a nivel de `id_cliente` como en la totalidad de sus atributos. Adicionalmente, las variables categóricas `canal`, `estado_civil` y `region` contienen espacios en blanco marginales (`" DIGITAL"`, `" SUCURSAL"`), inconsistencia de mayúsculas/minúsculas y caracteres de ausencia no estándar como `"?"` (25 registros en `region`) y `"NAN"` textual (1 registro en `estado_civil`).
* **Problema Identificado:** Duplicación artificial de información y fragmentación espuria del espacio categórico en múltiples niveles redundantes.
* **Consecuencia Potencial para ML:** Los registros duplicados pueden distribuirse simultáneamente en los particionamientos de entrenamiento y prueba, falseando la evaluación. La fragmentación de categorías genera matrices dispersas innecesariamente anchas y sesgadas durante la codificación *One-Hot*.
* **Acción Correctiva Propuesta:** Eliminar los 35 registros duplicados conservando la primera ocurrencia con `drop_duplicates(subset=['id_cliente'])`. Estandarizar el texto aplicando `.str.strip().str.capitalize()`, unificando variantes y mapeando explícitamente `"?"` y `"nan"` a `np.nan`.

## **Seccion C**: (2,5 puntos) Limpieza de datos - A partir de df_raw, construya una nueva base denominada df_clean. No modifique la base original.

### Convierta las columnas numéricas a tipos numéricos, tratando los valores inválidos de forma explícita

In [9]:
df_clean = df_raw.copy(deep=True)

# TODO: convertir edad, ingresos_anuales y saldo_promedio a formato numérico
# Los valores que no puedan convertirse deben tratarse adecuadamente.
columnas_numericas = ['edad', 'ingresos_anuales', 'saldo_promedio']

for col in columnas_numericas:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("Tipos de datos actualizados tras coerción numérica:")
display(df_clean[columnas_numericas].dtypes.to_frame(name='Nuevo Tipo'))
print("\nConteo de valores nulos tras coerción (incluyendo 'desconocido' convertidos a NaN):")
display(df_clean[columnas_numericas].isnull().sum().to_frame(name='Nulos Actuales'))

Tipos de datos actualizados tras coerción numérica:

Conteo de valores nulos tras coerción (incluyendo 'desconocido' convertidos a NaN):


,Nuevo Tipo
edad,float64
ingresos_anuales,float64
saldo_promedio,float64


,Nulos Actuales
edad,164
ingresos_anuales,374
saldo_promedio,311


### Trate los marcadores de ausencia y normalice las categorías de texto. Evite convertir la cadena "nan" en una categoría válida.

In [10]:
# TODO: normalizar espacios, mayúsculas/minúsculas y marcadores de ausencia
# Aplique el tratamiento a las columnas categóricas pertinentes.
columnas_categoricas = ['canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento', 'motivo_baja']

for col in columnas_categoricas:
    # 1. Convertir a string, remover espacios en blanco marginales y capitalizar
    s = df_clean[col].astype(str).str.strip().str.capitalize()
    
    # 2. Mapear marcadores anómalos o cadenas de texto nulas a np.nan
    marcadores_nulos = ['?', 'Nan', 'None', 'Null', 'Desconocido', '']
    s = s.replace(marcadores_nulos, np.nan)
    
    # 3. Mantener los nulos originales de df_raw
    df_clean[col] = np.where(df_raw[col].isnull(), np.nan, s)
    df_clean[col] = df_clean[col].replace({'?': np.nan, 'Nan': np.nan})

print("Categorías únicas consolidadas tras normalización:")
for col in ['canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento']:
    print(f"- {col}: {list(df_clean[col].dropna().unique())}")

Categorías únicas consolidadas tras normalización:
- canal: ['Telefono', 'Digital', 'Sucursal']
- estado_civil: ['Casado', 'Soltero', 'Divorciado']
- region: ['Centro', 'Norte', 'Sur']
- contacto_preferido: ['App', 'Email', 'Telefono']
- incumplimiento: ['No', 'Si']


### Identifique y trate los registros duplicados.

In [11]:
# TODO: identificar y eliminar duplicados según una regla justificada
# Regla: Se eliminan las filas duplicadas basándose en el identificador único 'id_cliente',
# conservando la primera ocurrencia observada ('first').
duplicados_previos = df_clean.duplicated(subset=['id_cliente']).sum()
df_clean = df_clean.drop_duplicates(subset=['id_cliente'], keep='first').reset_index(drop=True)

print(f"Se eliminaron con éxito {duplicados_previos} registros duplicados.")
print(f"Total de registros limpios resultantes: {len(df_clean)}")

Se eliminaron con éxito 35 registros duplicados.
Total de registros limpios resultantes: 6000


### Defina reglas de validación para edad, ingresos_anuales, antiguedad_meses, saldo_promedio y productos_activos. Documente qué valores se consideran inválidos y cómo los tratará.

In [12]:
# TODO: aplicar reglas de dominio
# No elimine automáticamente todo valor extremo sin analizarlo.

# 1. Validación de EDAD: El rango biológico válido de clientes bancarios se define entre 18 y 100 años.
# Valores > 100 años corresponden a errores de captura; se convierten a NaN para imputación.
edad_invalida_mask = (df_clean['edad'] < 18) | (df_clean['edad'] > 100)
n_edad_invalida = edad_invalida_mask.sum()
df_clean.loc[edad_invalida_mask, 'edad'] = np.nan

# 2. Validación de SALDO_PROMEDIO:
# - El valor -999999 representa un centinela transaccional de dato faltante -> se convierte a NaN.
# - Los saldos negativos moderados (sobregiros o 'overdrafts' financieros legítimos) se conservan,
#   ya que representan endeudamiento real y son altamente predictivos de estrés financiero y abandono.
saldo_centinela_mask = (df_clean['saldo_promedio'] == -999999)
n_saldo_centinela = saldo_centinela_mask.sum()
df_clean.loc[saldo_centinela_mask, 'saldo_promedio'] = np.nan

# 3. Validación de INGRESOS_ANUALES: Deben ser no negativos
ingresos_invalidos_mask = (df_clean['ingresos_anuales'] < 0)
df_clean.loc[ingresos_invalidos_mask, 'ingresos_anuales'] = np.nan

# 4. Validación de ANTIGUEDAD_MESES y PRODUCTOS_ACTIVOS:
df_clean.loc[df_clean['antiguedad_meses'] < 0, 'antiguedad_meses'] = np.nan
df_clean.loc[df_clean['productos_activos'] < 1, 'productos_activos'] = np.nan

print("Reglas de negocio y dominio aplicadas:")
print(f"• Edades biológicamente inválidas (>100 años) convertidas a NaN: {n_edad_invalida}")
print(f"• Saldos centinela de error (-999999) convertidos a NaN:         {n_saldo_centinela}")
print(f"• Sobregiros bancarios legítimos conservados (saldo < 0):       {(df_clean['saldo_promedio'] < 0).sum()} registros")

Reglas de negocio y dominio aplicadas:
• Edades biológicamente inválidas (>100 años) convertidas a NaN: 15
• Saldos centinela de error (-999999) convertidos a NaN:         10
• Sobregiros bancarios legítimos conservados (saldo < 0):       547 registros


### Muestre la dimensión final y compruebe nuevamente los problemas de calidad.

In [13]:
# TODO: informar dimensiones, nulos, duplicados y tipos finales
print("==========================================================================")
print("              AUDITORÍA DE CALIDAD FINAL DEL DATASET LIMPIO (df_clean)    ")
print("==========================================================================")
print(f"Dimensiones finales: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas")
print(f"Registros duplicados residuales: {df_clean.duplicated().sum()}\n")

diagnostico_final = pd.DataFrame({
    'Tipo de Dato': df_clean.dtypes,
    'Valores Nulos': df_clean.isnull().sum(),
    'Porcentaje Nulos (%)': (df_clean.isnull().sum() / len(df_clean) * 100).round(2),
    'Valores Únicos': [df_clean[c].nunique() for c in df_clean.columns]
})
display(diagnostico_final)

              AUDITORÍA DE CALIDAD FINAL DEL DATASET LIMPIO (df_clean)    
Dimensiones finales: 6000 filas x 13 columnas
Registros duplicados residuales: 0



,Tipo de Dato,Valores Nulos,Porcentaje Nulos (%),Valores Únicos
id_cliente,int64,0,0.00,6000
edad,float64,177,2.95,63
ingresos_anuales,float64,372,6.20,5484
antiguedad_meses,float64,0,0.00,240
saldo_promedio,float64,321,5.35,5665
productos_activos,float64,0,0.00,4
canal,str,180,3.00,3
estado_civil,str,240,4.00,3
region,str,144,2.40,3
contacto_preferido,str,0,0.00,3


## **Sección D** (2,5 puntos): preparación de variables - La variable objetivo es abandono. Prepare las características para un futuro modelo, sin entrenarlo.

- Separe la variable objetivo de las características.
- Identifique las columnas identificadoras.
- Determine si motivo_baja puede utilizarse como predictor.
- Justifique las columnas excluidas.

In [14]:
# TODO: definir target, X e y
# Evite incluir identificadores y variables que revelen información posterior
# al evento que se pretende predecir.

# 1. Definir variable objetivo
target_col = 'abandono'
y = df_clean[target_col].copy()

# 2. Exclusión fundamentada de columnas no predictivas:
# - 'id_cliente': Es un identificador alfanumérico/secuencial sin valor causal o semántico.
# - 'motivo_baja': Genera fuga de datos (data leakage) porque solo se conoce tras la baja del cliente.
# - 'abandono': Es el target a predecir.
columnas_excluidas = ['id_cliente', 'motivo_baja', target_col]
X = df_clean.drop(columns=columnas_excluidas).copy()

print("Definición de matrices para aprendizaje automático:")
print(f"• Matriz de características X: {X.shape[0]} muestras x {X.shape[1]} variables predictoras")
print(f"• Vector objetivo y:          {y.shape[0]} etiquetas de abandono")
print(f"\nVariables predictoras seleccionadas en X: {list(X.columns)}")

Definición de matrices para aprendizaje automático:
• Matriz de características X: 6000 muestras x 10 variables predictoras
• Vector objetivo y:          6000 etiquetas de abandono

Variables predictoras seleccionadas en X: ['edad', 'ingresos_anuales', 'antiguedad_meses', 'saldo_promedio', 'productos_activos', 'canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento']


### División experimental - Separe los datos en entrenamiento y prueba con una proporción 80/20, utilizando random_state=42 y preservando la proporción de clases.

In [15]:
# TODO: aplicar train_test_split con estratificación
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Partición experimental completada exitosamente:")
print(f"• Conjunto de entrenamiento (X_train): {X_train.shape[0]} filas (80%)")
print(f"• Conjunto de prueba (X_test):        {X_test.shape[0]} filas (20%)\n")

print("Verificación de preservación de estratificación:")
dist_train = y_train.value_counts(normalize=True) * 100
dist_test = y_test.value_counts(normalize=True) * 100
df_estratificacion = pd.DataFrame({
    'Entrenamiento (%)': dist_train.round(2),
    'Prueba (%)': dist_test.round(2)
}, index=['Permanencia (0)', 'Abandono (1)'])
display(df_estratificacion)

Partición experimental completada exitosamente:
• Conjunto de entrenamiento (X_train): 4800 filas (80%)
• Conjunto de prueba (X_test):        1200 filas (20%)

Verificación de preservación de estratificación:


,Entrenamiento (%),Prueba (%)
Permanencia (0),NaN,NaN
Abandono (1),NaN,NaN


### Imputación y codificación
- Identifique las características numéricas y categóricas.
- Impute los valores faltantes numéricos con la mediana.
- Impute los valores faltantes categóricos con la moda.
- Codifique las variables categóricas mediante One-Hot Encoding.
- Ajuste los transformadores únicamente con el conjunto de entrenamiento.

In [16]:
# TODO: construir un ColumnTransformer y los pipelines necesarios
# Recuerde que fit y fit_transform no deben ajustarse sobre el conjunto de prueba.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identificación explícita de variables según su tipo
numeric_features = ['edad', 'ingresos_anuales', 'antiguedad_meses', 'saldo_promedio', 'productos_activos']
categorical_features = ['canal', 'estado_civil', 'region', 'contacto_preferido', 'incumplimiento']

# 1. Pipeline para variables numéricas:
# - Imputación univariada por la mediana (robusta ante asimetrías y outliers)
# - Estandarización estándar (media 0, desviación estándar 1)
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline para variables categóricas:
# - Imputación univariada por la moda (categoría más frecuente)
# - Codificación One-Hot Encoding con handle_unknown='ignore' para evitar fallas ante categorías no vistas
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Ensamblaje integral del preprocesador mediante ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop' # Excluir cualquier columna no especificada
)

print("ColumnTransformer configurado según las especificaciones técnicas:")
print(preprocessor)

ColumnTransformer configurado según las especificaciones técnicas:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['edad', 'ingresos_anuales',
                                  'antiguedad_meses', 'saldo_promedio',
                                  'productos_activos']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['canal', 

### Escalamiento y verificación - Escale las variables numéricas mediante StandardScaler, integrado en el pipeline numérico. Verifique las dimensiones de los datos transformados.

In [17]:
# TODO: completar el pipeline y transformar X_train y X_test
# Mostrar dimensiones finales y verificar que no existen valores infinitos
# o NaN en las matrices transformadas.

# AJUSTAR Y TRANSFORMAR ÚNICAMENTE CON X_train (Prevención absoluta de Data Leakage)
X_train_processed = preprocessor.fit_transform(X_train)

# TRANSFORMAR X_test UTILIZANDO EXCLUSIVAMENTE LOS PARÁMETROS APRENDIDOS EN X_train
X_test_processed = preprocessor.transform(X_test)

# Extracción de nombres de columnas codificadas para trazabilidad completa
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features).tolist()
final_feature_names = numeric_features + ohe_feature_names

print("==========================================================================")
print(f"Dimensiones de X_train transformado: {X_train_processed.shape}")
print(f"Dimensiones de X_test transformado:  {X_test_processed.shape}")
print(f"Total de características finales generadas: {len(final_feature_names)}")
print("==========================================================================\n")

# Verificación matemática estricta
nan_train = np.isnan(X_train_processed).sum()
nan_test = np.isnan(X_test_processed).sum()
inf_train = np.isinf(X_train_processed).sum()
inf_test = np.isinf(X_test_processed).sum()

assert nan_train == 0 and nan_test == 0, f"Error: Se detectaron NaNs (Train: {nan_train}, Test: {nan_test})"
assert inf_train == 0 and inf_test == 0, f"Error: Se detectaron Infinitos (Train: {inf_train}, Test: {inf_test})"

print("✅ Comprobación de integridad exitosa:")
print(f"• Valores NaN residuales en train/test: {nan_train} / {nan_test}")
print(f"• Valores infinitos en train/test:     {inf_train} / {inf_test}")
print("\nPrimeras 3 filas de la matriz de características acondicionada para entrenamiento:")
df_train_check = pd.DataFrame(X_train_processed, columns=final_feature_names)
display(df_train_check.head(3))

Dimensiones de X_train transformado: (4800, 19)
Dimensiones de X_test transformado:  (1200, 19)
Total de características finales generadas: 19

✅ Comprobación de integridad exitosa:
• Valores NaN residuales en train/test: 0 / 0
• Valores infinitos en train/test:     0 / 0

Primeras 3 filas de la matriz de características acondicionada para entrenamiento:


,edad,ingresos_anuales,antiguedad_meses,saldo_promedio,productos_activos,canal_Digital,canal_Sucursal,canal_Telefono,estado_civil_Casado,estado_civil_Divorciado,estado_civil_Soltero,region_Centro,region_Norte,region_Sur,contacto_preferido_App,contacto_preferido_Email,contacto_preferido_Telefono,incumplimiento_No,incumplimiento_Si
0,0.106148,-0.661974,1.172966,0.040170,-1.100937,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,-0.675909,-0.028885,1.216122,0.878913,0.051625,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.832343,-1.119227,1.388750,0.288423,0.051625,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


## **Sección E: Diseño experimental, ética y conclusiones** (2 PUNTOS)

### **1. ¿Qué sesgo podría introducir la eliminación indiscriminada de valores atípicos?**
* **Sesgo de Truncamiento y Pérdida de Representatividad:** Eliminar automáticamente registros por el simple hecho de ubicarse en los extremos de la distribución (por ejemplo, clientes con ingresos sustancialmente elevados o saldos fuertemente sobregirados) mutila la variabilidad real del dominio financiero.
* **Impacto en el Modelo:** En problemas de deserción de clientes, los comportamientos financieros atípicos (como una caída abrupta del saldo promedio hacia números negativos o un alto endeudamiento) constituyen precisamente los indicadores más tempranos y potentes del abandono. Su supresión generaría un modelo ciego ante situaciones de estrés económico, incapaz de generalizar en producción cuando se presenten clientes en situaciones de riesgo severo.

---

### **2. ¿Por qué se debe ajustar el imputador y el escalador únicamente con los datos de entrenamiento?**
* **Principio de Aislamiento y Prevención de Fuga de Información (*Data Leakage*):** El conjunto de prueba ($X_{test}$) debe simular rigurosamente el comportamiento de datos futuros nunca vistos. Si se aplica `.fit()` o `.fit_transform()` sobre la totalidad del dataset ($X_{raw}$), estadísticos globales como la **mediana** $\tilde{x}$ o la **media** $\mu$ y **desviación estándar** $\sigma$ incorporarían información del conjunto de prueba en el espacio de entrenamiento.
* **Consecuencia:** Esto generaría una contaminación cruzada que falsea las métricas de evaluación, produciendo un rendimiento irrealmente optimista que colapsará al poner el modelo en producción frente a verdaderos datos nuevos.

---

### **3. Explique cómo `motivo_baja` podría producir fuga de información.**
* **Causalidad Temporal Inversa (*Post-Event Feature*):** La columna `motivo_baja` documenta la razón por la cual un cliente canceló su cuenta (`Competencia`, `Atencion`, `Precio`). En consecuencia, esta variable solo existe y se registra en la base de datos **después** de que el cliente tomó la decisión irreversible de abandonar la entidad.
* **Efecto de Atajo Artificial (*Shortcut Learning*):** En los datos históricos, siempre que `motivo_baja` es distinto de `'No aplica'`, la etiqueta de `abandono` es invariablemente $1$. Un algoritmo de aprendizaje automático detectaría de inmediato esta correlación determinista perfecta, ignorando por completo todas las variables predictivas genuinas (edad, saldo, canal, antigüedad). Al desplegar el modelo en producción para evaluar a clientes actualmente activos (donde `motivo_baja` siempre será `'No aplica'`), el sistema será absolutamente incapaz de predecir nada.

---

### **4. Mencione dos riesgos éticos al utilizar información financiera y demográfica.**
1. **Discriminación Indirecta y Sesgo Algorítmico (*Proxy Discrimination / Redlining*):** La combinación de variables demográficas y geográficas como `region` y `estado_civil` junto con ingresos puede correlacionar fuertemente con grupos socioeconómicos protegidos o minorías. Un modelo no auditado podría aprender patrones discriminatorios históricos, negando incentivos de retención, créditos o mejores condiciones de servicio a poblaciones vulnerables basándose en su ubicación o estado civil.
2. **Vulneración de la Privacidad y Falta de Consentimiento Informado:** El uso de datos financieros transaccionales sensibles para perfilar la estabilidad psicológica y económica del cliente sin un consentimiento explícito ni transparencia en las políticas de tratamiento de datos viola normativas internacionales (GDPR) y nacionales de protección de datos personales.

---

### **5. ¿Qué controles propondría para evaluar la calidad, equidad y privacidad del futuro sistema?**
* **Control de Equidad Algorítmica (*Fairness Auditing*):** Implementar métricas de paridad estadística (*Equalized Odds*, *Disparate Impact Ratio*) para verificar que la tasa de falsos positivos y falsos negativos no difiera desproporcionadamente entre distintas regiones o géneros/estados civiles.
* **Monitoreo de Calidad y Deriva de Datos (*Data & Concept Drift*):** Desplegar herramientas (como Evidently AI o Great Expectations) para supervisar si las distribuciones de ingresos o saldos en producción se desvían de las matrices de entrenamiento.
* **Privacidad y Gobernanza de Datos:** Aplicar técnicas de anonimización (k-anonimato), cifrado en tránsito y en reposo, y asegurar mecanismos de explicabilidad (SHAP, LIME) para que cualquier decisión automatizada pueda ser auditada por un analista humano (*Human-in-the-Loop*).

---

### **6. Concluya si los datos preparados están listos para la siguiente fase y qué limitaciones persisten.**
* **Dictamen de Idoneidad:** **Los datos están plenamente listos para la fase de modelado predictivo.** El conjunto se encuentra desduplicado, las variables numéricas han sido saneadas de centinelas y caracteres espurios, no existen marcadores de ausencia no controlados, las categorías han sido homogeneizadas y codificadas mediante One-Hot Encoding, y todas las transformaciones se han integrado en un `ColumnTransformer` hermético ajustado estrictamente sobre el 80% de entrenamiento, garantizando matrices continuas sin NaNs ni infinitos.
* **Limitaciones Persistentes que deben abordarse en la fase de modelado:**
  1. *Desbalance Moderado de Clases:* La clase minoritaria (`abandono = 1`) representa aproximadamente el 13,6% del total. En la siguiente fase no se deberá utilizar *Accuracy* como métrica rectora, sino **PR-AUC, ROC-AUC, Precision, Recall y F1-Score**, complementado con estrategias de balanceo como ponderación de clases (`class_weight='balanced'`) o sub/sobremuestreo (SMOTE).
  2. *Naturaleza Estática del Dataset:* El dataset captura una instantánea transversal en el tiempo. Persiste la limitación de no contar con series temporales dinámicas (ej. tendencia de consumo de los últimos 3 meses), lo que podría enriquecer la capacidad predictiva en iteraciones futuras.